In [43]:
import torch
import torchvision as tv
import albumentations as A

from tqdm import tqdm

In [3]:
train_dataset = tv.datasets.FashionMNIST(root=".", train=True, transform=tv.transforms.ToTensor(), download=True)
test_dataset = tv.datasets.FashionMNIST(root=".", train=False, transform=tv.transforms.ToTensor(), download=True)

100%|██████████| 26.4M/26.4M [00:31<00:00, 836kB/s] 
100%|██████████| 29.5k/29.5k [00:00<00:00, 261kB/s]
100%|██████████| 4.42M/4.42M [00:05<00:00, 837kB/s] 
100%|██████████| 5.15k/5.15k [00:00<00:00, 6.03MB/s]


In [4]:
batch_size = 64

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [41]:
class MyCNN(torch.nn.Module):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.act = torch.nn.ReLU()
        self.pool = torch.nn.MaxPool2d((2, 2), 2)

        self.conv1 = torch.nn.Conv2d(1, 16, (2, 2), stride=1, padding=1)
        self.conv2 = torch.nn.Conv2d(16, 32, (2, 2), stride=1, padding=1)
        self.conv3 = torch.nn.Conv2d(32, 64, (2, 2), stride=1, padding=1)

        self.flatten = torch.nn.Flatten()
        self.linear1 = torch.nn.Linear(1024, 1024)
        self.linear2 = torch.nn.Linear(1024, 10)


    def forward(self, x):
        x = self.conv1(x)
        x = self.pool(x)
        x = self.act(x)

        x = self.conv2(x)
        x = self.pool(x)
        x = self.act(x)

        x = self.conv3(x)
        x = self.pool(x)
        x = self.act(x)

        x = self.flatten(x)
        x = self.linear1(x)
        x = self.act(x)
        x = self.linear2(x)
        return x


lr = 0.01
model = MyCNN()
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

In [45]:
EPOCHS = 10


for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    train_progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for x, y in train_progress:

        optimizer.zero_grad()
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(y_pred.data, 1)
        train_total += y.size(0)
        train_correct += (predicted == y).sum().item()

    avg_train_loss = train_loss / len(train_loader)
    train_acc = 100.0 * train_correct / train_total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for x, y in test_loader:
            y_pred = model(x)
            loss = loss_fn(y_pred, y)

            val_loss += loss.item()
            _, predicted = torch.max(y_pred.data, 1)
            val_total += y.size(0)
            val_correct += (predicted == y).sum().item()

    avg_val_loss = val_loss / len(test_loader)
    val_acc = 100.0 * val_correct / val_total

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%")

Epoch 1/10 | Train Loss: 0.4959, Train Acc: 81.35% | Val Loss: 0.3953, Val Acc: 84.97%


Epoch 2/10 | Train Loss: 0.3428, Train Acc: 87.28% | Val Loss: 0.4032, Val Acc: 85.54%


Epoch 3/10 | Train Loss: 0.3130, Train Acc: 88.31% | Val Loss: 0.3305, Val Acc: 87.95%


Epoch 4/10 | Train Loss: 0.3002, Train Acc: 88.70% | Val Loss: 0.3228, Val Acc: 88.04%


Epoch 5/10 | Train Loss: 0.2916, Train Acc: 89.26% | Val Loss: 0.3499, Val Acc: 87.23%


Epoch 6/10 | Train Loss: 0.2784, Train Acc: 89.48% | Val Loss: 0.3463, Val Acc: 87.06%


Epoch 7/10 | Train Loss: 0.2774, Train Acc: 89.54% | Val Loss: 0.3458, Val Acc: 87.42%


Epoch 8/10 | Train Loss: 0.2689, Train Acc: 89.97% | Val Loss: 0.3156, Val Acc: 88.47%


Epoch 9/10 | Train Loss: 0.2650, Train Acc: 90.11% | Val Loss: 0.3959, Val Acc: 86.26%


Epoch 10/10 | Train Loss: 0.2600, Train Acc: 90.22% | Val Loss: 0.3031, Val Acc: 89.26%
